In [36]:
%pip install agent-framework python-dotenv 

Note: you may need to restart the kernel to use updated packages.


In [37]:
import os
from dotenv import load_dotenv


load_dotenv()

True

In [38]:
from agent_framework.openai import OpenAIChatCompletionClient

In [39]:
#define local Ollama (OpenAI-compatible) router settings
OLLAMA_BASE_URL = os.getenv("OLLAMA_ENDPOINT", "http://localhost:11434/v1")
OLLAMA_MODEL = os.getenv("OLLAMA_CHAT_MODEL", "llama3.1")

# Ollama ignores the key's value, but the OpenAI SDK requires a non-empty string
OLLAMA_API_KEY = os.getenv("OLLAMA_API_KEY", "ollama")


In [40]:
client = OpenAIChatCompletionClient(
    api_key=OLLAMA_API_KEY,
    base_url=OLLAMA_BASE_URL,
    model=OLLAMA_MODEL
)

In [41]:
from dataclasses import dataclass, asdict
from typing import Optional


@dataclass
class RootAgentMemory:
    """D1 — Root Agent Memory: name, email, phone."""

    first_name: Optional[str] = None
    last_name: Optional[str] = None
    email: Optional[str] = None
    phone_number: Optional[str] = None

    def missing_fields(self) -> list[str]:
        return [
            field
            for field in (
                "first_name",
                "last_name",
                "email",
                "phone_number",
            )
            if not getattr(self, field)
        ]

    def is_complete(self) -> bool:
        return not self.missing_fields()

    def as_dict(self) -> dict:
        return asdict(self)


# One store per guest session.
# In a real deployment, create one instance per conversation/session.
root_agent_memory = RootAgentMemory()

In [43]:
# Root Agent tools
import re
from typing import Annotated

from agent_framework import tool


_EMAIL_RE = re.compile(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")
_PHONE_RE = re.compile(r"^\+?[0-9()\-\s]{7,20}$")


@tool
def record_first_and_last_name(
    first_name: Annotated[
        str,
        "The guest's first name, as given by the guest."
    ],
    last_name: Annotated[
        str,
        "The guest's last name, as given by the guest."
    ],
) -> str:
    """Save the guest's first and last name to D1 (Root Agent Memory)."""

    root_agent_memory.first_name = first_name.strip()
    root_agent_memory.last_name = last_name.strip()

    return (
        f"Saved name: "
        f"{root_agent_memory.first_name} "
        f"{root_agent_memory.last_name}"
    )


@tool
def record_email(
    email: Annotated[
        str,
        "The guest's email address, as given by the guest."
    ],
) -> str:
    """Validate and save the guest's email address to D1 (Root Agent Memory)."""

    email = email.strip()

    if not _EMAIL_RE.match(email):
        return (
            f"'{email}' doesn't look like a valid email address. "
            "Ask the guest to double-check it and try again."
        )

    root_agent_memory.email = email

    return f"Saved email: {email}"


@tool
def record_phone_number(
    phone_number: Annotated[
        str,
        "The guest's phone number, as given by the guest."
    ],
) -> str:
    """Validate and save the guest's phone number to D1 (Root Agent Memory)."""

    phone_number = phone_number.strip()

    if not _PHONE_RE.match(phone_number):
        return (
            f"'{phone_number}' doesn't look like a valid phone number. "
            "Ask the guest to double-check it and try again."
        )

    root_agent_memory.phone_number = phone_number

    return f"Saved phone number: {phone_number}"


@tool
def check_identity_status() -> str:
    """Check which identity fields are still missing from D1."""

    missing = root_agent_memory.missing_fields()

    if not missing:
        return (
            "All identity fields are recorded. "
            "Ready to hand off to the Coordinator Agent."
        )

    return f"Still missing: {', '.join(missing)}"


@tool
def handoff_to_coordinator_agent() -> str:
    """
    Route the conversation to the Coordinator Agent (P2)
    once identity collection is complete.

    P2 is not implemented in this notebook, so this is a stub.
    It confirms the handoff and returns the identity record
    P2 would receive.
    """

    if not root_agent_memory.is_complete():
        missing = ", ".join(
            root_agent_memory.missing_fields()
        )

        return f"Cannot hand off yet — still missing: {missing}."

    return (
        "Handoff to Coordinator Agent (P2) complete. "
        f"Identity payload: {root_agent_memory.as_dict()}"
    )


root_agent_tools = [
    record_first_and_last_name,
    record_email,
    record_phone_number,
    check_identity_status,
    handoff_to_coordinator_agent,
]

In [44]:
#root agent instrctions
ROOT_AGENT_INSTRUCTIONS = """
You are the Root Agent for a hotel booking assistant. You are the entry point of the
system — your only job is to collect the guest's identity, one step at a time, before
handing the conversation off to the Coordinator Agent. Do not answer booking, policy,
pricing, or payment questions yourself; that's the Coordinator Agent's job after handoff.

Follow this exact sequence, asking only one question per turn:

1. Ask for the guest's first name and last name. You may ask both in one question, or
   as two short questions if that reads more naturally. Once you have both, call
   record_first_and_last_name.

2. Ask for the guest's email address. Once given, call record_email. If the tool
   reports the email looks invalid, tell the guest and ask them to re-enter it.

3. Ask for the guest's phone number. Once given, call record_phone_number. If the tool
   reports the number looks invalid, tell the guest and ask them to re-enter it.

4. Call check_identity_status to confirm everything is recorded.

5. Once nothing is missing, call handoff_to_coordinator_agent, then tell the guest in
   one short, friendly sentence that you're connecting them to booking assistance now.

Rules:

- Be concise and friendly. Ask for exactly one piece of information at a time — do not
  ask for name, email, and phone all in a single message.
- Never fabricate or guess a value; only save what the guest actually provided.
- Never call handoff_to_coordinator_agent until check_identity_status confirms nothing
  is missing.
- Do not re-ask for a field that has already been recorded successfully.
""".strip()

In [45]:
from agent_framework import Agent


In [46]:
#create an agent 

root_agent = Agent(
    client=client,
    name="RootAgent",
    description="Entry point agent that collects guest identity (P1 in the hotel booking DFD).",
    instructions=ROOT_AGENT_INSTRUCTIONS,
    tools=root_agent_tools,
)

In [47]:
#test demo

import asyncio


async def run_scripted_demo() -> None:
    session = root_agent.create_session()

    guest_turns = [
        "Hi, I'd like to book a room.",
        "My name is Amara Perera.",
        "amara.perera@example.com",
        "+94 77 123 4567",
    ]

    for turn in guest_turns:
        print(f"Guest: {turn}")
        result = await root_agent.run(turn, session=session)
        print(f"RootAgent: {result.text}\n")

    print("--- D1 (Root Agent Memory) after the conversation ---")
    print(root_agent_memory.as_dict())


await run_scripted_demo()

Guest: Hi, I'd like to book a room.
RootAgent: Hello! I'd be happy to help you with that. To get started, may I have your first and last name, please?

Guest: My name is Amara Perera.
RootAgent: Thank you, Amara. What is your email address?

Guest: amara.perera@example.com
RootAgent: Thank you. And finally, may I have your phone number?

Guest: +94 77 123 4567
RootAgent: I'm connecting you to booking assistance now!

--- D1 (Root Agent Memory) after the conversation ---
{'first_name': 'Amara', 'last_name': 'Perera', 'email': 'amara.perera@example.com', 'phone_number': '+94 77 123 4567'}
